In [ ]:
# Robust Windows approach: run Julia as a separate process (no embedding / no PyCall needed)

import subprocess, tempfile, textwrap

julia_exe = r"C:\\Users\\nabil\\.julia\\juliaup\\julia-1.12.4+0.x64.w64.mingw32\\bin\\julia.exe"
code = textwrap.dedent("""
function add(a,b)
    return a + b
end
println(add(2,3))
""")

with tempfile.NamedTemporaryFile('w', suffix='.jl', delete=False, encoding='utf-8') as f:
    f.write(code)
    julia_file = f.name

proc = subprocess.run([julia_exe, julia_file], capture_output=True, text=True)
if proc.returncode != 0:
    raise RuntimeError(proc.stderr)

print(proc.stdout.strip())  # 5

5


### Call `MADBead.fit_E_z_MC` from Python (subprocess)
This uses the same robust approach: run Julia as an external process and exchange data via JSON.

**Inputs expected by `fit_E_z_MC`:**
- `nu_hz`: 1D array of frequencies (Hz)
- `z`: 1D array of z positions (same units you used in Julia, typically meters)
- `intdA_E_val`, `intdA_E_err`: 2D arrays shaped `(len(nu_hz), len(z))` representing `abs.(∫dA_E)` values and 1σ errors
- `p0_all`: dict mapping parameter name → `{val, err}` (numbers). This mirrors the Julia `Dict` of `Measurement`s.
- `keys_optim`, `keys_fixed`, `keys_helper`: lists of parameter names (strings)
- `N_mc`: number of Monte-Carlo samples

In [ ]:
import json
import subprocess
import tempfile
import textwrap
from pathlib import Path

import numpy as np


def _find_madbead_dir() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        candidate = p / 'dark-photon-analysis-main' / 'MADBead'
        if (candidate / 'MADBead.jl').exists():
            return candidate
    raise FileNotFoundError('Could not locate dark-photon-analysis-main/MADBead from current working directory.')


def _normalize_measurement_dict(d: dict) -> dict:
    # Accept either {val:.., err:..} or plain numbers.
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out[k] = {'val': float(v.get('val', 0.0)), 'err': float(v.get('err', 0.0))}
        else:
            out[k] = {'val': float(v), 'err': 0.0}
    return out


def madbead_fit_E_z_MC(
    nu_hz,
    z,
    intdA_E_val,
    intdA_E_err,
    p0_all: dict,
    keys_optim: list[str],
    keys_fixed: list[str],
    keys_helper: list[str],
    N_mc: int,
    julia_exe: str | None = None,
    madbead_dir: str | Path | None = None,
):
    """
    Calls MADBead.fit_E_z_MC in Julia and returns a dict:
      { 'f': [..], 'samples': {param: [[...N_mc...], ... per frequency] } }

    Note: this expects *magnitudes* (abs) for intdA_E inputs, like the Julia code uses via abs.(∫dA_E).
    """
    nu_hz = np.asarray(nu_hz, dtype=float).ravel()
    z = np.asarray(z, dtype=float).ravel()
    intdA_E_val = np.asarray(intdA_E_val, dtype=float)
    intdA_E_err = np.asarray(intdA_E_err, dtype=float)

    if intdA_E_val.shape != (nu_hz.size, z.size):
        raise ValueError(f'intdA_E_val must have shape {(nu_hz.size, z.size)} but got {intdA_E_val.shape}')
    if intdA_E_err.shape != intdA_E_val.shape:
        raise ValueError('intdA_E_err must have the same shape as intdA_E_val')

    if madbead_dir is None:
        madbead_dir = _find_madbead_dir()
    else:
        madbead_dir = Path(madbead_dir)

    if julia_exe is None:
        # Prefer whatever is on PATH; fall back to the juliaup-managed binary used above.
        import shutil
        julia_exe = shutil.which('julia') or r'C:\Users\nabil\.julia\juliaup\julia-1.12.4+0.x64.w64.mingw32\bin\julia.exe'

    payload = {
        'nu': nu_hz.tolist(),
        'z': z.tolist(),
        'intdA_E': {'val': intdA_E_val.tolist(), 'err': intdA_E_err.tolist()},
        'p0_all': _normalize_measurement_dict(p0_all),
        'keys_optim': list(keys_optim),
        'keys_fixed': list(keys_fixed),
        'keys_helper': list(keys_helper),
        'N_mc': int(N_mc),
    }

    runner = textwrap.dedent('''
        import JSON
        using Measurements
        using DataFrames

        infile = ARGS[1]
        outfile = ARGS[2]
        madbead_dir = ARGS[3]

        include(joinpath(madbead_dir, "MADBead.jl"))
        using .MADBead

        payload = JSON.parsefile(infile)
        ν = Float64.(payload["nu"])
        z = Float64.(payload["z"])

        vals = payload["intdA_E"]["val"]
        errs = payload["intdA_E"]["err"]
        ∫dA_E = [measurement(Float64(vals[i][j]), Float64(errs[i][j])) for i in 1:length(ν), j in 1:length(z)]

        p0_in = payload["p0_all"]
        p0_all = Dict{String,Any}()
        for (k, v) in p0_in
            p0_all[string(k)] = measurement(Float64(v["val"]), Float64(v["err"]))
        end

        keys_optim = String.(payload["keys_optim"])
        keys_fixed = String.(payload["keys_fixed"])
        keys_helper = String.(payload["keys_helper"])
        N_mc = Int(payload["N_mc"])

        df = MADBead.fit_E_z_MC(ν, z, ∫dA_E, p0_all, keys_optim, keys_fixed, keys_helper, N_mc)

        out = Dict{String,Any}()
        out["f"] = collect(df[!, "f"])
        samples = Dict{String,Any}()
        for k in keys_optim
            samples[k] = [collect(df[i, k]) for i in 1:nrow(df)]
        end
        out["samples"] = samples

        open(outfile, "w") do io
            JSON.print(io, out)
        end
    ''')

    with tempfile.TemporaryDirectory() as td:
        td = Path(td)
        in_json = td / 'input.json'
        out_json = td / 'output.json'
        runner_jl = td / 'runner.jl'

        in_json.write_text(json.dumps(payload), encoding='utf-8')
        runner_jl.write_text(runner, encoding='utf-8')

        proc = subprocess.run(
            [str(julia_exe), str(runner_jl), str(in_json), str(out_json), str(madbead_dir)],
            capture_output=True,
            text=True,
        )
        if proc.returncode != 0:
            raise RuntimeError(proc.stderr or proc.stdout)

        return json.loads(out_json.read_text(encoding='utf-8'))


# --- Example (you must replace these dummy numbers with your real data) ---
# nu_hz = np.array([22e9])
# z = np.linspace(0.0, 0.03, 25)
# intdA_E_val = np.abs(np.sin(z))[None, :] + 0.1
# intdA_E_err = 0.02 * np.ones_like(intdA_E_val)
#
# p0_all = {
#     'E_0': {'val': 1.0, 'err': 1.0},
#     'z_m': {'val': 0.03, 'err': 0.0},
#     'σ_m': {'val': 3.5e7, 'err': 0.0},
#     'n_disk': {'val': 1.0, 'err': 0.0},
#     'd_v_1': {'val': 0.01, 'err': 1e-4},
#     'd_d_1': {'val': 0.01, 'err': 0.0},
#     'ϵ_d_1': {'val': 9.4, 'err': 0.0},
#     'tanD_d_1': {'val': 1e-4, 'err': 0.0},
#     'r_b': {'val': 1e-3, 'err': 0.0},
#     'ϵ_b': {'val': 9.4, 'err': 0.0},
# }
# keys_optim = ['E_0', 'd_v_1']
# keys_helper = ['n_disk']
# keys_fixed = [k for k in p0_all.keys() if k not in keys_optim + keys_helper]
# out = madbead_fit_E_z_MC(nu_hz, z, intdA_E_val, intdA_E_err, p0_all, keys_optim, keys_fixed, keys_helper, N_mc=10)
# print(out['f'])
# print(list(out['samples'].keys()))

In [ ]:
# Quick check that Julia can load MADBead and the symbol exists.
# (A full call to fit_E_z_MC needs physically sensible inputs; random dummy data can make LsqFit fail.)

madbead_dir = _find_madbead_dir()

import shutil
julia_exe_check = shutil.which('julia') or r'C:\Users\nabil\.julia\juliaup\julia-1.12.4+0.x64.w64.mingw32\bin\julia.exe'

check_runner = textwrap.dedent('''
    madbead_dir = ARGS[1]
    include(joinpath(madbead_dir, "MADBead.jl"))
    using .MADBead
    println(isdefined(MADBead, :fit_E_z_MC))
''')

with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    runner_jl = td / 'check.jl'
    runner_jl.write_text(check_runner, encoding='utf-8')
    proc = subprocess.run([str(julia_exe_check), str(runner_jl), str(madbead_dir)], capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr or proc.stdout)
    print('MADBead.fit_E_z_MC exists:', proc.stdout.strip())

# To actually run the fit, use madbead_fit_E_z_MC(...) with your real nu/z/∫dA_E/p0_all from the measurement.

MADBead.fit_E_z_MC exists: true


### Fit 1D model to reduced data (`ETs_reduced`, `z_reduced`)
This cell builds `intdA_E_val` and `intdA_E_err` from `ETs_reduced` and calls `fit_E_z_MC`.
Make sure the variables below exist (after running the main data-processing cell):
- `ETs_reduced` (shape: Z x Nf)
- `z_reduced` (shape: Z)
- `frequencies` (shape: Nf, in Hz)
- `p0_all`, `keys_optim`, `keys_fixed`, `keys_helper` (see reference in [dark-photon-analysis-main/4_Analysis_Mirror_Position.ipynb](dark-photon-analysis-main/4_Analysis_Mirror_Position.ipynb) or [dark-photon-analysis-main/5_Analysis_Boostfactor_Determination_3Disks.ipynb](dark-photon-analysis-main/5_Analysis_Boostfactor_Determination_3Disks.ipynb)).

In [ ]:
# --- p0_all and keys_* defaults from the Julia notebooks ---
# Choose the parameter set used in the Julia notebooks:
fit_mode = 'boostfactor'  # 'mirror' (4_Analysis_Mirror_Position) or 'boostfactor' (5_Analysis_Boostfactor_Determination_3Disks)
force_reinit = True  # set True to overwrite existing p0_all/keys_*

# If z_region is available (from Julia notebooks), use it to determine n_disk
if 'z_region' in globals():
    n_disk_val = int(len(z_region) - 1)
else:
    # Mirror-position notebook often uses n_disk = 0 (mirror-only)
    n_disk_val = 0 if fit_mode == 'mirror' else 3
    print(f'z_region not found; defaulting n_disk={n_disk_val}. Adjust if needed.')

# Common parameter values from the Julia notebooks
eps_b = {'val': 9.23, 'err': 0.2}
r_b = {'val': 2.93e-3 / 2, 'err': 15e-6 / 2}
sigma_m = {'val': 5.0e7, 'err': 1.0e7}
eps_d = {'val': 9.3, 'err': 0.1}
tanD_d = {'val': 1e-5, 'err': 1e-6}
d_d = {'val': 1e-3, 'err': 0.05e-3}
E0_0 = {'val': 1.0, 'err': 1.0}
z_m_0 = {'val': 675.551e-3, 'err': 1e-3}  # mechanical estimate (FM504_M notebook)

if force_reinit or 'p0_all' not in globals() or 'keys_optim' not in globals():
    p0_all = {
        'E_0': E0_0,
        'z_m': z_m_0,
        'σ_m': sigma_m,
        'n_disk': {'val': float(n_disk_val), 'err': 0.0},
        'r_b': r_b,
        'ϵ_b': eps_b,
    }

    if fit_mode == 'mirror':
        # matches 4_Analysis_Mirror_Position.ipynb
        keys_optim = ['E_0', 'z_m']
        keys_helper = ['n_disk']
        keys_fixed = [k for k in p0_all.keys() if k not in keys_optim + keys_helper]
    elif fit_mode == 'boostfactor':
        # matches 5_Analysis_Boostfactor_Determination_3Disks.ipynb
        if n_disk_val < 1:
            raise ValueError('n_disk must be >= 1 for boostfactor mode.')
        # d_v_i defaults from the notebook (three disks)
        d_v_defaults = [8.265e-3, 9.813e-3, 9.813e-3]
        if n_disk_val > len(d_v_defaults):
            raise ValueError('n_disk exceeds default d_v_i list. Provide your own d_v_i values.')
        for i in range(n_disk_val):
            p0_all[f'd_v_{i+1}'] = {'val': d_v_defaults[i], 'err': 0.1e-3}
            p0_all[f'd_d_{i+1}'] = d_d
            p0_all[f'ϵ_d_{i+1}'] = eps_d
            p0_all[f'tanD_d_{i+1}'] = tanD_d
        keys_optim = [f'd_v_{i+1}' for i in range(n_disk_val)]
        keys_optim.insert(0, 'E_0')
        keys_helper = ['n_disk']
        keys_fixed = [k for k in p0_all.keys() if k not in keys_optim + keys_helper]
    else:
        raise ValueError(
)

    print('Initialized p0_all/keys_* from Julia notebook defaults.')
else:
    print('Using existing p0_all/keys_* already defined in the notebook.')

z_region not found; defaulting n_disk=3. Adjust if needed.
Initialized p0_all/keys_* from Julia notebook defaults.


In [ ]:
# Diagnostics: check reduced data scale and finiteness
if 'ETs_reduced' in globals() and 'z_reduced' in globals() and 'frequencies' in globals():
    z_arr = np.asarray(z_reduced).reshape(-1)
    E_arr = np.asarray(ETs_reduced)
    print('z_reduced: min/max', np.nanmin(z_arr), np.nanmax(z_arr), 'len', z_arr.size)
    print('ETs_reduced: shape', E_arr.shape)
    print('ETs_reduced finite:', np.isfinite(E_arr).all())
    print('ETs_reduced nan/inf count:', np.isnan(E_arr).sum(), np.isinf(E_arr).sum())
    print('frequencies: min/max', float(np.nanmin(frequencies)), float(np.nanmax(frequencies)))
else:
    print('ETs_reduced/z_reduced/frequencies not found yet.')

z_reduced: min/max 2.0 35.0 len 4
ETs_reduced: shape (4, 1001)
ETs_reduced finite: True
ETs_reduced nan/inf count: 0 0
frequencies: min/max 18000000000.0 24000000000.0


In [ ]:
# Diagnostics: inspect p0_all and key lists
if 'p0_all' in globals():
    print('p0_all keys:', sorted(p0_all.keys()))
    bad = []
    for k, v in p0_all.items():
        if isinstance(v, dict):
            val = v.get('val', None)
            err = v.get('err', None)
            if val is None or not np.isfinite(val):
                bad.append((k, 'val', val))
            if err is None or not np.isfinite(err):
                bad.append((k, 'err', err))
        else:
            if not np.isfinite(v):
                bad.append((k, 'val', v))
    if bad:
        print('Non-finite in p0_all:', bad)
    else:
        print('p0_all values look finite.')
else:
    print('p0_all not defined yet.')

if 'keys_optim' in globals():
    print('keys_optim:', keys_optim)
if 'keys_fixed' in globals():
    print('keys_fixed:', keys_fixed)
if 'keys_helper' in globals():
    print('keys_helper:', keys_helper)

p0_all keys: ['E_0', 'd_d_1', 'd_d_2', 'd_d_3', 'd_v_1', 'd_v_2', 'd_v_3', 'n_disk', 'r_b', 'tanD_d_1', 'tanD_d_2', 'tanD_d_3', 'z_m', 'σ_m', 'ϵ_b', 'ϵ_d_1', 'ϵ_d_2', 'ϵ_d_3']
p0_all values look finite.
keys_optim: ['E_0', 'd_v_1', 'd_v_2', 'd_v_3']
keys_fixed: ['z_m', 'σ_m', 'r_b', 'ϵ_b', 'd_d_1', 'ϵ_d_1', 'tanD_d_1', 'd_d_2', 'ϵ_d_2', 'tanD_d_2', 'd_d_3', 'ϵ_d_3', 'tanD_d_3']
keys_helper: ['n_disk']


In [ ]:
# --- Fit 1D model to reduced data ---
required_vars = ['ETs_reduced', 'z_reduced', 'frequencies', 'p0_all', 'keys_optim', 'keys_fixed', 'keys_helper']
missing = [v for v in required_vars if v not in globals()]
if missing:
    print('Missing variables:', missing)
    print('Run the main processing cell and define p0_all/keys_* before fitting.')
else:
    ETs_reduced_arr = np.asarray(ETs_reduced)
    z_reduced_arr = np.asarray(z_reduced).reshape(-1)
    nu_hz = np.asarray(frequencies, dtype=float).reshape(-1)

    if ETs_reduced_arr.ndim != 2:
        raise ValueError(f'ETs_reduced must be 2D (Z x Nf). Got shape {ETs_reduced_arr.shape}')
    if z_reduced_arr.size != ETs_reduced_arr.shape[0]:
        raise ValueError('z_reduced length must match ETs_reduced first dimension (Z).')
    if nu_hz.size != ETs_reduced_arr.shape[1]:
        raise ValueError('frequencies length must match ETs_reduced second dimension (Nf).')

    # sort by z to keep monotonic ordering (recommended for fitting)
    order = np.argsort(z_reduced_arr)
    z_sorted = z_reduced_arr[order]
    ETs_sorted = ETs_reduced_arr[order, :]

    # Heuristic: if z is in mm (values > 1), convert to meters for MADBead
    if np.nanmax(z_sorted) > 1.0:
        print('Detected z in mm; converting to meters for Julia fit.')
        z_sorted = z_sorted * 1e-3

    # drop z rows with non-finite values across any frequency
    z_mask = np.all(np.isfinite(ETs_sorted), axis=1)
    z_sorted = z_sorted[z_mask]
    ETs_sorted = ETs_sorted[z_mask, :]

    if z_sorted.size < 3:
        raise ValueError('Not enough finite z points after filtering.')

    # Build |∫dA E| arrays: shape (Nf, Z)
    intdA_E_val = np.abs(ETs_sorted).T

    # drop frequency rows with any non-finite values across z
    f_mask = np.all(np.isfinite(intdA_E_val), axis=1)
    intdA_E_val = intdA_E_val[f_mask, :]
    nu_hz = nu_hz[f_mask]

    if nu_hz.size < 1:
        raise ValueError('No valid frequencies after filtering non-finite data.')

    # Set a conservative relative error if you don't have one (adjust as needed)
    rel_err = 0.05
    min_err = (np.nanmax(intdA_E_val) * 1e-6) if np.nanmax(intdA_E_val) > 0 else 1e-12
    intdA_E_err = np.maximum(rel_err * intdA_E_val, min_err)

    # Run the Monte Carlo fit
    out = madbead_fit_E_z_MC(
        nu_hz, z_sorted,
        intdA_E_val, intdA_E_err,
        p0_all, keys_optim, keys_fixed, keys_helper,
        N_mc=1000,
    )

    print('fit_E_z_MC done. Keys in result:', list(out['samples'].keys()))
    # Example: convert samples for a parameter into an array (freq x N_mc)
    # samples_E0 = np.asarray(out['samples']['E_0'])

Detected z in mm; converting to meters for Julia fit.


plot the fitted E_z field over z position and draw the disks and mirror into the plot. Also show in the plot the original data points of E_z used for the fit.